# 04 - Evaluacion sobre Test

Carga ambos modelos entrenados y evalua sobre `splits/test/`. Genera matrices de confusion y reporta F1 por clase.


In [ ]:
!pip install -q tensorflow scikit-learn matplotlib seaborn

In [ ]:
import tensorflow as tf
import numpy as np
import json
from pathlib import Path
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, f1_score, precision_score, recall_score,
)
import matplotlib.pyplot as plt
import seaborn as sns

OUT = Path("./outputs")
SPLIT = Path("./splits")

m1 = tf.keras.models.load_model(OUT / "model1_binary.keras")
m2 = tf.keras.models.load_model(OUT / "model2_pathogen.keras")
print("Modelos cargados OK")


In [ ]:
import sys
sys.path.insert(0, "..")

import albumentations as A
import numpy as np
from pathlib import Path
from PIL import Image
from tensorflow.keras.utils import Sequence
from tensorflow.keras.applications.efficientnet import preprocess_input

VAL_AUG = A.Compose([])

class LeafSequence(Sequence):
    def __init__(self, directory, img_size=(224, 224), batch_size=32, class_mode="binary"):
        self.img_size = img_size
        self.batch_size = batch_size
        self.class_mode = class_mode
        self.samples = []
        self.class_indices = {}
        classes = sorted(p.name for p in Path(directory).iterdir() if p.is_dir())
        for i, cls in enumerate(classes):
            self.class_indices[cls] = i
            for ext in ("*.jpg", "*.jpeg", "*.png", "*.bmp"):
                for fp in (Path(directory) / cls).glob(ext):
                    self.samples.append((str(fp), i))
        self.n = len(self.samples)
        self.classes = np.array([s[1] for s in self.samples])

    def __len__(self):
        return max(1, (self.n + self.batch_size - 1) // self.batch_size)

    def __getitem__(self, i):
        batch = self.samples[i * self.batch_size:(i + 1) * self.batch_size]
        imgs, labels = [], []
        for fp, label in batch:
            img = np.array(Image.open(fp).convert("RGB").resize(self.img_size))
            img = preprocess_input(img.astype(np.float32))
            imgs.append(img)
            labels.append(label)
        X = np.stack(imgs)
        if self.class_mode == "binary":
            Y = np.array(labels, dtype=np.float32)
        else:
            Y = np.eye(len(self.class_indices))[labels]
        return X, Y


In [ ]:
test1 = LeafSequence(
    SPLIT / "test" / "clasificacion_binaria",
    batch_size=32, class_mode="binary",
)
preds1 = (m1.predict(test1, verbose=1) > 0.5).astype(int).flatten()
reales1 = test1.classes[:len(preds1)]
print("=== Modelo 1 ===")
print(classification_report(reales1, preds1, target_names=list(test1.class_indices.keys()), digits=4))

fig, ax = plt.subplots(figsize=(5, 4))
cm1 = confusion_matrix(reales1, preds1)
sns.heatmap(cm1, annot=True, fmt="d", cmap="Blues",
            xticklabels=list(test1.class_indices.keys()),
            yticklabels=list(test1.class_indices.keys()), ax=ax)
ax.set_title("Modelo 1 - Confusion")
plt.tight_layout()
plt.savefig(OUT / "cm_m1.png", dpi=120)
plt.show()


In [ ]:
test2 = LeafSequence(
    SPLIT / "test" / "clasificacion_patogeno",
    batch_size=32, class_mode="categorical",
)
probs2 = m2.predict(test2, verbose=1)
preds2 = np.argmax(probs2, axis=1)
reales2 = test2.classes[:len(preds2)]
names2 = list(test2.class_indices.keys())
print("=== Modelo 2 ===")
print(classification_report(reales2, preds2, target_names=names2, digits=4))

fig, ax = plt.subplots(figsize=(8, 7))
cm2 = confusion_matrix(reales2, preds2)
sns.heatmap(cm2, annot=True, fmt="d", cmap="Blues",
            xticklabels=names2, yticklabels=names2, ax=ax)
ax.set_title("Modelo 2 - Confusion (5 clases)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(OUT / "cm_m2.png", dpi=120)
plt.show()


In [ ]:
metrics = {
    "m1": {
        "accuracy": float(accuracy_score(reales1, preds1)),
        "f1": float(f1_score(reales1, preds1, zero_division=0)),
        "precision": float(precision_score(reales1, preds1, zero_division=0)),
        "recall": float(recall_score(reales1, preds1, zero_division=0)),
    },
    "m2": {
        "accuracy": float(accuracy_score(reales2, preds2)),
        "f1_macro": float(f1_score(reales2, preds2, average="macro", zero_division=0)),
        "f1_per_class": {
            n: float(s) for n, s in zip(
                names2, f1_score(reales2, preds2, average=None, zero_division=0)
            )
        },
    },
}
with open(OUT / "training_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print("\n=== Verificacion de objetivos ===")
print(f"M1 accuracy: {metrics['m1']['accuracy']:.4f}  (objetivo >= 0.85) {'OK' if metrics['m1']['accuracy'] >= 0.85 else 'FALLA'}")
print(f"M1 F1:       {metrics['m1']['f1']:.4f}  (objetivo >= 0.85) {'OK' if metrics['m1']['f1'] >= 0.85 else 'FALLA'}")
print(f"M2 accuracy: {metrics['m2']['accuracy']:.4f}  (objetivo >= 0.75) {'OK' if metrics['m2']['accuracy'] >= 0.75 else 'FALLA'}")
print(f"M2 macro F1: {metrics['m2']['f1_macro']:.4f}  (objetivo >= 0.70) {'OK' if metrics['m2']['f1_macro'] >= 0.70 else 'FALLA'}")
for cls, f1v in metrics['m2']['f1_per_class'].items():
    print(f"  {cls:20s}: F1={f1v:.4f}  {'OK' if f1v >= 0.65 else 'REVISAR'}")
